In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import time
import mlflow

from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import f1_score, accuracy_score
from dotenv import load_dotenv


from src.resnet import ResNet18
from src.inference import extract_features
from src.pca import PCATransformer
from src.draw_figures import *
from src.dataloader import build_excluded_dataset, get_dataset_config
from src.paths import DATA_DIR, FIGURES_DIR, MODELS_DIR, OUTPUT_DIR, ensure_dir

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlflow.set_experiment("TotalVariance")

seed = 42
target = 9
epoch = 15

%matplotlib inline

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

### Готовим данные:

In [ ]:
dataset_name = "cifar10"
dataset_config = get_dataset_config(dataset_name)
all_class_ids = list(range(dataset_config.num_classes))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_config.mean, dataset_config.std)
])


generator = torch.Generator().manual_seed(seed)
output_dir = ensure_dir(OUTPUT_DIR / dataset_name / str(target)) 


### То, на чем обучаем resnet, строим pca
train_dataset = build_excluded_dataset(
    name=dataset_name,
    root=str(DATA_DIR),
    exclude_class=[target],
    train=True, 
    transform=transform
)

train_size = int(0.8 * len(train_dataset))
val_size = int(0.15 * len(train_dataset))

# выборка для построение PCA пространства, для тренеровки модели разделения
test_size = (len(train_dataset) - train_size - val_size) // 2 
# выбока, которая уйдет в финальный тест нового пайплайна
new_pipeline_test_size = len(train_dataset) - train_size - val_size - test_size


train, val, test, new_pipeline_test = random_split(
    train_dataset, 
    [train_size, val_size, test_size, new_pipeline_test_size], 
    generator=generator
)

train_loader = DataLoader(train, batch_size=64, shuffle=True, num_workers=2, generator=generator)
val_loader   = DataLoader(val, batch_size=64, shuffle=False, num_workers=2, generator=generator)
test_loader = DataLoader(test, batch_size=64, shuffle=False, num_workers=2, generator=generator)

### То, на чем делаем холостой (все классы будует неверные) предикт ради feature layer, на чем обучаем и проводим тесты lda
target_dataset = build_excluded_dataset(
    name=dataset_name,
    root=str(DATA_DIR),
    exclude_class=[class_id for class_id in all_class_ids if class_id != target],
    train=True, 
    transform=transform
)

target_train_size = int(0.85 * len(target_dataset))
target_test_size = (len(target_dataset) - target_train_size)
other = len(target_dataset) - target_train_size - target_test_size

train_target, test_target, other = random_split(
    target_dataset, 
    [target_train_size, target_test_size, other]
)

train_target_loader = DataLoader(train_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)
test_target_loader = DataLoader(test_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)

### То, на чем пробуем новый пайплайн
new_pipeline_loader = DataLoader(new_pipeline_test+test_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)


In [ ]:
load_dotenv()
mlflow.start_run(run_name=f"exclude_class_{target}_epoch{epoch}")

mlflow.log_params({
    "target": target,
    "seed": seed,
})

In [ ]:
class_distr_path = class_distribution(
    new_pipeline_loader,
    str(output_dir / "new_pipeline_class_distribution.png")
)


mlflow.log_artifact(class_distr_path)

### Подгружаем модель и извлекаем признаки

In [ ]:
resnet = ResNet18(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        learning_rate=0.001,
        num_classes=dataset_config.num_classes,
        input_channels=dataset_config.input_channels,
        # num_epochs=8,
        weights=MODELS_DIR / f"resnet18_{dataset_name}_without{target}_epoch_{epoch}.pth"
    )

# history = resnet.train()

In [ ]:
test_cifar9_features, test_cifar9_preds, test_cifar9_gt_preds = extract_features(resnet.model, test_loader, device)
test_cifar9_iscorrect = test_cifar9_gt_preds == test_cifar9_preds

mask = test_cifar9_gt_preds == test_cifar9_preds
test_cifar9_CR = test_cifar9_features[mask == True]
test_cifar9_WR = test_cifar9_features[mask == False]

test_cifar9_f1_score =  f1_score(test_cifar9_gt_preds, test_cifar9_preds, average="macro")
test_cifar9_f1_score_list = f1_score(test_cifar9_gt_preds, test_cifar9_preds, average=None)

print("f1-мера (глобальные TP и FP): {}, \nПо каждому классу: {}".format(
        test_cifar9_f1_score,
        test_cifar9_f1_score_list
    )
)
class_ids = [class_id for class_id in all_class_ids if class_id != target]
class_names = [str(class_id) for class_id in class_ids]

conf_martix_path = plot_confusion_matrix(
    test_cifar9_preds, 
    test_cifar9_gt_preds, 
    class_names=class_names,
    labels=class_ids,
    title="Confusion Matrix for test sample (without target class)",
    path=str(output_dir / "train" / "confusion_matrix.png")
)

# print(conf_martix_path)

conf_norm_martix_path = plot_confusion_matrix(
    test_cifar9_preds, 
    test_cifar9_gt_preds, 
    class_names=class_names,
    labels=class_ids,
    title="Confusion Matrix for test sample (without target class)",
    path=str(output_dir / "train" / "confusion_norm_matrix.png"),
    normalize="true"
)


In [ ]:
mlflow.log_artifact(conf_martix_path, artifact_path="Test_loader") 
mlflow.log_artifact(conf_norm_martix_path, artifact_path="Test_loader") 

mlflow.log_metric("test_f1_macro", test_cifar9_f1_score)

for cls, f1 in zip(class_names, test_cifar9_f1_score_list):
        mlflow.log_metric(f"test_f1_class_{cls}", f1)

In [ ]:
train_target_features, train_target_preds, train_target_gt_preds = extract_features(resnet.model, train_target_loader, device)

mask = train_target_gt_preds == train_target_preds
train_target_CR = train_target_features[mask == True]  
train_target_WR = train_target_features[mask == False] 

len(train_target_CR), len(train_target_WR)

plot_confusion_matrix(train_target_preds, train_target_gt_preds)

### Строим новый пайплайн

In [ ]:

def new_pipeline(features: np.array, preds: np.array, sep_model, proj):
    features_p = proj.transform(features)


    # se_predict =  np.where(lda.predict_proba(features_p)[:, 1] > thresh, 1, 0)
    sep_predict = sep_model.predict(features_p)

    predict = []
    for i in range(sep_predict.shape[0]):
        if sep_predict[i]:
            predict.append(target)
        else:
            predict.append(preds[i])

    return np.array(predict)

def print_metric(preds, gt_preds):
    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
    new_pipeline_loader_f1_score_macro = f1_score(gt_preds, preds, average="macro")
    new_pipeline_loader_f1_score_micro = f1_score(gt_preds, preds, average="micro")
    new_pipeline_loader_f1_score_list = f1_score(gt_preds, preds, average=None)
    new_pipeline_loader_acc = accuracy_score(gt_preds, preds)

    print("f1-мера (глобальные TP и FP): {} \nf1-score (усреднее): {}" \
    "\nf1-score (по каждому классу): {} \nacc: {}".format(
        new_pipeline_loader_f1_score_macro,
        new_pipeline_loader_f1_score_micro,
        new_pipeline_loader_f1_score_list,
        new_pipeline_loader_acc
        )
    )

    return new_pipeline_loader_f1_score_macro, new_pipeline_loader_f1_score_list

In [ ]:
models = {
    "lda": LinearDiscriminantAnalysis(n_components=1),

    "linearSVC": LinearSVC(
        dual=False
    ),

    "logisticRegression": LogisticRegression(
        solver='liblinear'
    ),

    "randomForestClassifier": RandomForestClassifier(
        n_estimators=50,
        max_depth=5,
        random_state=seed,
        n_jobs=-1
    ),

    "xgbClassifier": XGBClassifier(
        n_estimators=20,
        max_depth=4,
        learning_rate=0.05,
        random_state=seed,
        n_jobs=-1
    )
}

In [ ]:
pca_full = PCATransformer(
    n_components=100,
    whiten=True
)

pca_full.fit(test_cifar9_features)

In [ ]:
explained_variance_first_component = round(pca_full.pca.explained_variance_ratio_[0], 3)
stop = round(explained_variance_first_component*100)

mlflow.log_metric("explained_variance_first_component", explained_variance_first_component)
explained_variance_first_component

In [ ]:
features, preds, gt_preds = extract_features(resnet.model, new_pipeline_loader, device)

In [ ]:
results = []
baseline_f1_macro = test_cifar9_f1_score

for variance in range(98, stop-1, -1):
    print(f"\nVariance = {0.99 - variance/100}")

    _, _, tail_variance = pca_full.select_variance_range(
        variance/100,
        0.99
    )

    # tail features
    test_cifar9_CR_tail = pca_full.transform(test_cifar9_CR)
    test_cifar9_WR_tail = pca_full.transform(test_cifar9_WR)
    train_target_p = pca_full.transform(train_target_features)

    print("Tail shape:", test_cifar9_CR_tail.shape)

    # visualization
    # plot_projection(
    #     test_cifar9_CR_tail,
    #     test_cifar9_WR_tail,
    #     path=str(output_dir / "pca" / f"proj_variance{tail_variance}.png")
    # )

    # dataset for separator
    X_pca = np.vstack([
        test_cifar9_CR_tail,
        test_cifar9_WR_tail,
        train_target_p
    ])

    y_pca = np.hstack([
        np.zeros(len(test_cifar9_CR_tail) + len(test_cifar9_WR_tail)),
        np.ones(len(train_target_p))
    ])

    # train + evaluate all models
    for model_name, model in models.items():

        print(f"Training {model_name}")

        model.fit(X_pca, y_pca)

        preds_pca = new_pipeline(
            features,
            preds,
            model,
            pca_full
        )

        f1_macro, f1_list = print_metric(
            preds_pca,
            gt_preds
        )

        power = np.nan if baseline_f1_macro == 0 else f1_macro / baseline_f1_macro

        results.append({
            "tail_variance": 0.99 - variance/100,
            "model": model_name,
            "f1_macro": f1_macro,
            "power": power,
        })


In [ ]:
df = pd.DataFrame(results)

print(df.head())


In [ ]:
df.to_csv(output_dir / "data.csv", index=False)

mlflow.log_artifact(str(output_dir / "data.csv"))

In [ ]:

path_f1 = plot_variance_vs_metric(
    df,
    metric_col="f1_macro",
    title=f"tail_variance vs F1 (target={target})",
    path=str(output_dir / "variance_vs_f1.png")
)

path_power = plot_variance_vs_metric(
    df,
    metric_col="power",
    title=f"tail_variance vs Power (target={target})",
    path=str(output_dir / "variance_vs_power.png")
)


In [ ]:
mlflow.log_artifact(path_f1, "total_tail_variance")
mlflow.log_artifact(path_power, "total_tail_variance")


In [ ]:
mlflow.end_run()